In [2]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, count, when, sum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from snowflake.snowpark import Session
from credentials import params
session = Session.builder.configs(params).create()

df_snowpark = session.table("HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA")
df = session.sql("SELECT * FROM HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA").to_pandas()

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_id                 35000 non-null  str    
 1   city                        35000 non-null  str    
 2   locality                    35000 non-null  str    
 3   locality_tier               35000 non-null  str    
 4   property_type               35000 non-null  str    
 5   bhk                         35000 non-null  int8   
 6   bathrooms                   35000 non-null  int8   
 7   balconies                   35000 non-null  int8   
 8   built_up_area               35000 non-null  int16  
 9   carpet_area                 35000 non-null  int16  
 10  floor_number                35000 non-null  int8   
 11  total_floors                35000 non-null  int8   
 12  floor_category              35000 non-null  str    
 13  facing                      35000 non-null

In [7]:
df.describe()

,bhk,bathrooms,balconies,built_up_area,carpet_area,floor_number,total_floors,property_age,parking_spaces,security_score,gym_available,swimming_pool,power_backup,lift_available,maintenance_fee_monthly,distance_to_city_center_km,distance_to_metro_km,nearby_schools,nearby_hospitals,price_in_lakhs
count,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.00000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000,35000.000000
mean,3.151829,3.365343,1.681971,1573.101686,1321.418886,9.33900,14.443086,6.504486,1.919514,5.999220,0.414286,0.299029,0.684657,0.840686,3981.710486,11.541657,2.509520,3.120657,1.906086,166.772213
std,1.214986,1.329136,1.232055,669.493721,563.581272,8.70655,8.817221,6.880724,1.451722,1.601971,0.492605,0.457839,0.464659,0.365974,1637.896999,5.452530,2.477522,1.898308,1.467227,104.714704
min,1.000000,1.000000,0.000000,300.000000,240.000000,0.00000,1.000000,0.000000,0.000000,0.100000,0.000000,0.000000,0.000000,0.000000,500.000000,4.000000,0.100000,0.000000,0.000000,10.000000
25%,2.000000,2.000000,1.000000,1046.750000,878.000000,3.00000,8.000000,2.000000,1.000000,4.900000,0.000000,0.000000,0.000000,1.000000,2682.000000,6.800000,0.700000,2.000000,1.000000,86.127500
50%,3.000000,3.000000,2.000000,1587.000000,1330.000000,7.00000,12.000000,4.000000,2.000000,5.900000,0.000000,0.000000,1.000000,1.000000,3667.000000,11.000000,1.800000,3.000000,2.000000,148.000000
75%,4.000000,4.000000,3.000000,2131.000000,1784.000000,14.00000,18.000000,9.000000,3.000000,7.100000,1.000000,1.000000,1.000000,1.000000,5180.000000,14.300000,3.500000,4.000000,3.000000,226.790000
max,5.000000,6.000000,4.000000,3056.000000,2666.000000,39.00000,45.000000,40.000000,5.000000,10.000000,1.000000,1.000000,1.000000,1.000000,8859.000000,45.000000,15.000000,10.000000,8.000000,737.590000


# Train/Test split

In [115]:
testset_cap = 5000

In [116]:
len_testset = session.sql(
    "SELECT COUNT(1) FROM HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET"
).collect()[0][0]

In [117]:
len_testset

0

In [81]:
testset_cap = 5000

len_testset = session.sql(
    "SELECT COUNT(1) FROM HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET"
).collect()[0][0]

if len_testset >= testset_cap:
    
    query = f"""SELECT * EXCLUDE ("bhk" ,"bathrooms", "built_up_area"),
    FROM HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEAN_DATA""" #sostituirla con lo stream: HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEANDATA_STREAM
    # aggiungere: WHERE METADATA$ACTION = 'INSERT'

    dml_query = f"""INSERT INTO HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET ({query})"""

    session.sql(dml_query).collect()

else:

    # CREATE THE TEMPORARY TABLE CONTAINING DATA + TRAIN/TEST SPLIT *************************************************************
    query = f"""SELECT * EXCLUDE ("bhk" ,"bathrooms", "built_up_area"),
                ROW_NUMBER() OVER (ORDER BY RANDOM()) AS random_numb
                FROM HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEAN_DATA""" #sostituirla con lo stream: HOUSING_PRICE_PROJECT.CLEANING_LAYER.CLEANDATA_STREAM
        # aggiungere: WHERE METADATA$ACTION = 'INSERT'
        
    ddl_query = f"""CREATE OR REPLACE TEMPORARY TABLE HOUSING_PRICE_PROJECT.ML_LAYER.CLEANDATA_SPLIT_TEMP 
        AS {query};"""
        
    session.sql(ddl_query).collect()


    # COUNT OF MISSING ROWS TO REACH TESTSET_CAP **********************************************************************************
    tot_rows = session.sql(
            "SELECT COUNT(1) FROM HOUSING_PRICE_PROJECT.ML_LAYER.CLEANDATA_SPLIT_TEMP"
    ).collect()[0][0]
    
    nr_test_rows = round(tot_rows * 0.2)
    
    miss_row_testset = testset_cap - len_testset

    thresh = min(nr_test_rows, miss_row_testset)
    

    # INSERTING THE ROWS EITHER IN TEST OR TRAIN SET *******************************************************************************
    for tab, cond in [("TEST_SET", "<="), ("TRAIN_SET",">")]:
    
        query = f"""SELECT * EXCLUDE random_numb
                    FROM HOUSING_PRICE_PROJECT.ML_LAYER.CLEANDATA_SPLIT_TEMP
                    WHERE random_numb {cond} {thresh}"""
        
        dml_query = f"""INSERT INTO HOUSING_PRICE_PROJECT.ML_LAYER.{tab} ({query})"""
    
        session.sql(dml_query).collect()

else


In [124]:
session.sql("SELECT COUNT(1) FROM HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET").collect()

[Row(COUNT(1)=30000)]